In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split

In [2]:
model = joblib.load(
    "C:\\Users\\Asus\\OneDrive\\Desktop\\Explainable-Credit-Risk-System\\models\\best_xgb_model.joblib"
)

print(type(model))

<class 'xgboost.sklearn.XGBClassifier'>


In [3]:
df_encoded = pd.read_csv(
    "C:\\Users\\Asus\\OneDrive\\Desktop\\Explainable-Credit-Risk-System\\data\\processed\\preprocessed_v2_encoded.csv"
)

print(df_encoded.shape)

(307507, 209)


In [4]:
X = df_encoded.drop(
    "TARGET",
    axis=1
)

y = df_encoded["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(X_test.shape)

(61502, 208)


In [5]:
sample_index = 0

sample = X_test.iloc[[sample_index]]

sample.head()

,SK_ID_CURR,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,...,ORGANIZATION_TYPE_Trade: type 4,ORGANIZATION_TYPE_Trade: type 5,ORGANIZATION_TYPE_Trade: type 6,ORGANIZATION_TYPE_Trade: type 7,ORGANIZATION_TYPE_Transport: type 1,ORGANIZATION_TYPE_Transport: type 2,ORGANIZATION_TYPE_Transport: type 3,ORGANIZATION_TYPE_Transport: type 4,ORGANIZATION_TYPE_University,ORGANIZATION_TYPE_XNA
180592,309296,0,247500.0,497520.0,59175.0,450000.0,0.014464,-19654,-235.0,-13084.0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
probability = model.predict_proba(
    sample
)[0][1]

print(
    "Default Probability:",
    round(probability * 100, 2),
    "%"
)

Default Probability: 4.53 %


In [7]:
if probability < 0.20:
    risk_level = "Low Risk"
elif probability < 0.50:
    risk_level = "Medium Risk"
else:
    risk_level = "High Risk"

print("Risk Level:", risk_level)

Risk Level: Low Risk


In [22]:
query = f"""
Applicant has:

- Default probability: {round(probability*100, 2)}%
- Risk level: {risk_level}
- Credit income ratio: {sample_data['CREDIT_INCOME_RATIO']:.2f}
- Annuity income ratio: {sample_data['ANNUITY_INCOME_RATIO']:.2f}
- Age: {sample_data['AGE_YEARS']:.1f} years
- Employment duration: {sample_data['EMPLOYMENT_YEARS']:.1f} years

What lending policies, credit guidelines, and risk rules are relevant for this applicant?
"""

print(query)


Applicant has:

- Default probability: 4.53000020980835%
- Risk level: Low Risk
- Credit income ratio: 2.01
- Annuity income ratio: 0.24
- Age: 53.8 years
- Employment duration: 0.6 years

What lending policies, credit guidelines, and risk rules are relevant for this applicant?



In [23]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    persist_directory="C:\\Users\\Asus\\OneDrive\\Desktop\\Explainable-Credit-Risk-System\\rag\\vectordb",
    embedding_function=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [24]:
results = retriever.invoke(query)

for doc in results:
    print("=" * 60)
    print(doc.page_content)

Risk Classification Rules

Low Risk:

* Predicted default probability below 20 percent.
* Strong external credit scores.
* Stable employment and income.

Medium Risk:

* Predicted default probability between 20 percent and 50 percent.
* Moderate debt burden.
* Some financial risk indicators present.
Lending Policies

1. Applicants with a debt-to-income ratio greater than 40 percent should undergo manual review.

2. Applicants requesting loans above approved thresholds require additional verification.

3. Employment duration below one year may increase lending risk.
6. A combination of low external credit scores and high debt obligations increases default probability.

7. Younger applicants with limited financial history may require additional verification.

8. Credit risk assessments should combine financial, behavioral, and demographic indicators.


In [25]:
sample_data = X_test.iloc[sample_index]

print(sample_data[
    [
        "CREDIT_INCOME_RATIO",
        "ANNUITY_INCOME_RATIO",
        "AGE_YEARS",
        "EMPLOYMENT_YEARS"
    ]
])

CREDIT_INCOME_RATIO      2.010182
ANNUITY_INCOME_RATIO     0.239091
AGE_YEARS               53.846575
EMPLOYMENT_YEARS         0.643836
Name: 180592, dtype: float64


In [27]:
from google import genai
from config import GEMINI_API_KEY

client = genai.Client(
    api_key=GEMINI_API_KEY
)

# Build context from retrieved documents
context = "\n\n".join(
    [doc.page_content for doc in results]
)

prompt = f"""
You are an expert Credit Risk Analyst working for a financial institution.

Your task is to generate a professional credit risk assessment report using:

1. Machine Learning prediction results
2. Applicant financial information
3. Retrieved lending policies and risk guidelines

=================================================
APPLICANT INFORMATION
=================================================

Predicted Default Probability:
{round(probability * 100, 2)}%

Risk Level:
{risk_level}

Credit Income Ratio:
{sample_data['CREDIT_INCOME_RATIO']:.2f}

Annuity Income Ratio:
{sample_data['ANNUITY_INCOME_RATIO']:.2f}

Age:
{sample_data['AGE_YEARS']:.1f} years

Employment Duration:
{sample_data['EMPLOYMENT_YEARS']:.1f} years

=================================================
RETRIEVED POLICIES AND GUIDELINES
=================================================

{context}

=================================================
INSTRUCTIONS
=================================================

Generate a professional credit risk assessment report.

Rules:

- Do not invent applicant IDs.
- Do not invent dates.
- Do not invent information not present in the input.
- Use only the provided applicant information.
- Use the retrieved policies when explaining decisions.
- Identify both positive and negative risk indicators.
- Explain why the applicant received the assigned risk level.
- Mention any lending policy that applies to the applicant.
- Keep the report concise, professional, and suitable for a bank credit officer.

IMPORTANT:

CREDIT_INCOME_RATIO represents:

AMT_CREDIT / AMT_INCOME_TOTAL

Do not interpret it as Debt-to-Income Ratio unless explicitly stated.

Use only the feature definitions provided.

=================================================
REPORT FORMAT
=================================================

1. Risk Level

2. Prediction Summary

3. Positive Risk Indicators

4. Negative Risk Indicators

5. Relevant Lending Policies

6. Final Recommendation

"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

## Credit Risk Assessment Report

**1. Risk Level**
Low Risk

**2. Prediction Summary**
The applicant has a predicted default probability of 4.53%, which falls significantly below the 20 percent threshold for Low Risk classification as per the institution's Risk Classification Rules. This machine learning prediction forms the primary basis for categorizing the applicant as Low Risk.

**3. Positive Risk Indicators**
*   **Low Predicted Default Probability:** A predicted default probability of 4.53% indicates a very low likelihood of default.
*   **Assigned Low Risk Level:** The overall assessment aligns with a Low Risk classification.
*   **Age:** At 53.8 years, the applicant is a mature individual, which is often associated with greater financial stability and experience.
*   **Annuity Income Ratio:** A ratio of 0.24 suggests that the applicant's existing annuity payments represent a manageable proportion of their income.

**4. Negative Risk Indicators**
*   **Employment Duration:** Th